# Signisa — Phase 1 diagnostic (CPU, internet ON)

Both losses hit the same ~73% TAR@FAR5 wall -> suspect a shared cause in the data or a
val signer, not the loss. Attach TWO inputs: the `kaggle_prep` output (tensors) and the
ArcFace `kaggle_train` output (model.pt). Paths are discovered by glob — no hardcoded mounts.

Reports: TAR/EER by val participant (+ mirrored flags), TAR by sign ascending, the 100
worst genuine trials with the centroid that beat them, and genuine-score spread per signer.
Writes `diagnosis_report.md` + `worst_genuine.csv`.

In [ ]:
from pathlib import Path

inputs = Path("/kaggle/input")
index_csvs = sorted(inputs.glob("*/tensors/index.csv")) or sorted(inputs.rglob("index.csv"))
models = sorted(inputs.rglob("model.pt"))
assert len(index_csvs) == 1, f"attach exactly ONE prep output; found {index_csvs}"
assert len(models) == 1, f"attach exactly ONE train output; found {models}"
tensors_dir = index_csvs[0].parent
model_pt = models[0]
print("tensors:", tensors_dir)
print("model:  ", model_pt)

In [ ]:
!git clone -q https://github.com/dayan-battulga/signisa.git /kaggle/working/signisa-repo
%pip install -q /kaggle/working/signisa-repo

In [ ]:
import torch

from signisa.eval import run_evaluation
from signisa.models import load_checkpoint

model = load_checkpoint(model_pt)
cfg = model.cfg
print(f"loaded {cfg.loss} model, landmark_version {cfg.landmark_version}")

REPO = "/kaggle/working/signisa-repo"
metrics = run_evaluation(
    model, tensors_dir, f"{REPO}/data/meta/curriculum_db.json",
    f"{REPO}/data/meta/training_labels.json", cfg, "/kaggle/working", device="cpu")
print(f"recomputed: TAR@FAR5 {metrics['tar_at_far']:.1%}, "
      f"top-1 {metrics['top1_closed_set']:.1%}")

In [ ]:
import json

import pandas as pd

trials = metrics["trials"]
thr = metrics["global_far_threshold"]
label_of = {c["id"]: c["label"] for c in json.load(
    open(f"{REPO}/data/meta/training_labels.json"))["classes"]}

# (a + d) per-participant: TAR, mean EER, genuine spread, mirrored flag
pp = pd.DataFrame(metrics["per_participant"]).T
pp.index.name = "participant"
print(pp.round(3).to_string(), "\n")

# (b) per-sign TAR at the global threshold, ascending
g = trials[trials.genuine].copy()
g["hit"] = g.score >= thr
by_sign = (g.groupby("target")
             .agg(n=("score", "size"), tar=("hit", "mean"), median_score=("score", "median"))
             .rename(index=label_of).sort_values("tar"))
print("worst signs by TAR:\n", by_sign.head(15).round(3).to_string(), "\n")

# (c) 100 worst genuine trials with margin and the centroid that beat them
imp = trials[~trials.genuine]
best = imp.loc[imp.groupby("sequence_id").score.idxmax(),
               ["sequence_id", "target", "score"]]
best.columns = ["sequence_id", "beaten_by_id", "best_impostor"]
worst = g.merge(best, on="sequence_id", how="left")
worst["margin"] = worst.score - worst.best_impostor
worst["sign"] = worst.target.map(label_of)
worst["beaten_by"] = worst.beaten_by_id.map(label_of)
worst = worst.sort_values("score").head(100)
cols = ["sequence_id", "participant", "sign", "score", "margin", "beaten_by", "best_impostor"]
worst[cols].round(4).to_csv("/kaggle/working/worst_genuine.csv", index=False)

def md_table(df):
    header = "| " + " | ".join(str(c) for c in [df.index.name or ""] + list(df.columns)) + " |"
    sep = "|" + "---|" * (len(df.columns) + 1)
    rows = ["| " + " | ".join(str(v) for v in [i] + list(r)) + " |"
            for i, r in zip(df.index, df.round(3).values)]
    return "\n".join([header, sep] + rows)

report = "\n\n".join([
    "# Phase 1 diagnostic",
    f"Global (recomputed): TAR@FAR5 {metrics['tar_at_far']:.1%}, "
    f"top-1 {metrics['top1_closed_set']:.1%}, threshold {thr:.3f}, loss={cfg.loss}.",
    "## Per-participant (bad-signer / wrong-mirroring check)", md_table(pp),
    "## Bottom 20 signs by TAR", md_table(by_sign.head(20)),
    f"## Worst genuine trials (top 20 of {len(worst)}; full list in worst_genuine.csv)",
    md_table(worst[cols].set_index("sequence_id").head(20)),
])
Path("/kaggle/working/diagnosis_report.md").write_text(report + "\n")

spread = pp.tar_at_far.max() - pp.tar_at_far.min()
misses = by_sign[by_sign.tar <= by_sign.tar.quantile(0.2)].n.sum() / by_sign.n.sum()
print(f"summary: participant TAR spread {spread:.1%} "
      f"(concentrated signer problem if large); "
      f"bottom-20%-of-signs hold {misses:.1%} of genuine trials; "
      f"{(worst.margin < 0).sum()}/{len(worst)} worst genuines actually beaten by another centroid")

## Round 2 — orientation test, label-noise bound, sad deep-dive

Round 1 found mirrored val signers at ~82% TAR / 7.7% EER vs 68.7% / 56.7% unmirrored ->
suspected dominance-vote error. Cells below embed every val curriculum attempt BOTH ways
(stored orientation and canonical-space flip — exactly equivalent to mirroring the raw
sequence), bound TAR against suspected label noise, and dissect `sad`.
Writes `diagnosis2_report.md`.

In [ ]:
# Task 1 — orientation test: the decisive experiment for the dominance-vote hypothesis
import numpy as np
import torch

from signisa.data import ShardDataset, mirrored_stored
from signisa.preprocess.pipeline import with_derived_channels

trained = json.load(open("/kaggle/working/curriculum_db_trained.json"))
cent = {g: np.array(e["centroid"]) for g, e in trained["signs"].items()
        if e["centroid"] is not None and e["eer_threshold"] is not None}
id_of = {v: k for k, v in label_of.items()}
curriculum_ids = {id_of[g] for g in cent}

val_ds = ShardDataset(tensors_dir, participants=metrics["val_participants"])
sub = val_ds.index[val_ds.index.canonical_label_id.isin(curriculum_ids)].reset_index(drop=True)

def embed_rows(flip):
    chunks = []
    with torch.no_grad():
        for start in range(0, len(sub), 256):
            batch = []
            for r in sub.iloc[start:start + 256].itertuples():
                arr = val_ds.tensors[r.row].astype(np.float32)
                if flip:
                    arr = mirrored_stored(arr, val_ds.landmark_version)
                batch.append(with_derived_channels(arr[..., :3], arr[..., 3:],
                                                   val_ds.landmark_version))
            chunks.append(model.embedder(torch.from_numpy(np.stack(batch))).numpy())
    return np.concatenate(chunks)

emb_stored, emb_flipped = embed_rows(False), embed_rows(True)
own = np.stack([cent[label_of[l]] for l in sub.canonical_label_id])
orient = pd.DataFrame({
    "sequence_id": sub.sequence_id, "participant": sub.participant_id,
    "sign": sub.canonical_label_id.map(label_of), "mirrored_flag": sub.mirrored,
    "stored": (emb_stored * own).sum(1), "flipped": (emb_flipped * own).sum(1)})
orient["best"] = orient[["stored", "flipped"]].max(axis=1)

per_pid = orient.groupby("participant").agg(
    n=("stored", "size"), mirrored=("mirrored_flag", "mean"),
    med_stored=("stored", "median"), med_flipped=("flipped", "median"),
    tar_stored=("stored", lambda s: (s >= thr).mean()),
    tar_flipped=("flipped", lambda s: (s >= thr).mean()),
    tar_orient_max=("best", lambda s: (s >= thr).mean()),
    flip_wins=("stored", lambda s: (orient.loc[s.index, "flipped"] > s).mean()))
print(per_pid.round(3).to_string())

verdicts = []
for pid, r in per_pid.iterrows():
    # median delta is threshold-free — the TAR columns use the stored-orientation-calibrated
    # threshold and are descriptive only
    if r.med_flipped - r.med_stored > 0.10:
        verdicts.append(f"participant {pid}: flipping raises the genuine median "
                        f"{r.med_stored:.2f} -> {r.med_flipped:.2f} (TAR {r.tar_stored:.1%} -> "
                        f"{r.tar_flipped:.1%}) — the dominance vote chose the WRONG orientation")
    elif 0.35 < r.flip_wins < 0.65 and r.tar_orient_max - max(r.tar_stored, r.tar_flipped) > 0.05:
        verdicts.append(f"participant {pid}: neither single orientation wins "
                        f"(flip wins {r.flip_wins:.0%}) but orientation-max adds "
                        f"{r.tar_orient_max - max(r.tar_stored, r.tar_flipped):+.1%} — mixed-orientation clips")
print()
print("\n".join(verdicts) or "no participant shows an orientation problem")

In [ ]:
# Task 2 — clean-TAR bound (label-noise quantification)
g = trials[trials.genuine].copy()
imp = trials[~trials.genuine]
best = imp.loc[imp.groupby("sequence_id").score.idxmax(), ["sequence_id", "target", "score"]]
best.columns = ["sequence_id", "beaten_by_id", "best_impostor"]
g = g.merge(best, on="sequence_id", how="left")
g["sign"] = g.target.map(label_of)
g["beaten_by"] = g.beaten_by_id.map(label_of)

conf_sets = {s: set(e["confusables"]) for s, e in trained["signs"].items()}
beaten_in_conf = np.array([pd.notna(r.beaten_by) and r.beaten_by in conf_sets.get(r.sign, set())
                           for r in g.itertuples()])
g["suspect"] = ((g.score < 0.45) & pd.notna(g.beaten_by) & ~beaten_in_conf
                & (g.best_impostor > g.score))  # actually beaten, not just low

suspect_by_sign = (g.groupby("sign").suspect.agg(n_suspect="sum", rate="mean", n="count")
                     .sort_values("rate", ascending=False))
flagged = suspect_by_sign[suspect_by_sign.rate > 0.15]
clean_tar = float((g[~g.suspect].score >= thr).mean())
print(f"suspects: {int(g.suspect.sum())}/{len(g)} genuine trials ({g.suspect.mean():.1%}) "
      f"— PopSign documents ~19% noise")
print(f"TAR@FAR5 all {(g.score >= thr).mean():.1%} -> clean-TAR bound {clean_tar:.1%} "
      f"(same global threshold, suspect genuines excluded)")
print("flagged signs (>15% suspect):\n", flagged.round(3).to_string() if len(flagged) else " none")

# Task 3 — sad deep-dive
sad = g[g.sign == "sad"].sort_values("score").reset_index(drop=True)
if sad.empty:
    sad_hist_txt, sad_para = "(no sad trials in this run)", "(no sad trials in this run)"
else:
    buckets = pd.cut(sad.score, bins=np.arange(-1.0, 1.01, 0.1))
    sad_hist = sad.groupby(buckets, observed=True).agg(
        n=("score", "size"),
        beaten_by=("beaten_by", lambda s: ", ".join(f"{k}x{v}" for k, v in s.value_counts().head(3).items())))
    sad_hist_txt = sad_hist.to_string()
    print("\nsad genuine-score histogram:\n", sad_hist_txt)

    scores = sad.score.to_numpy()
    gaps = np.diff(scores)
    gi = int(gaps.argmax()) if len(gaps) else 0
    low, high = scores[:gi + 1], scores[gi + 1:]
    bimodal = len(gaps) > 0 and gaps[gi] > 0.15 and len(low) >= 5 and len(high) >= 5
    emb_of = dict(zip(orient.sequence_id, emb_stored))
    if bimodal:
        L = np.stack([emb_of[s] for s in sad.sequence_id[:len(low)] if s in emb_of])
        coh = float((L @ L.T)[np.triu_indices(len(L), 1)].mean()) if len(L) > 1 else float("nan")
        rivals = sad.beaten_by[:len(low)].value_counts()
        sad_para = (
            f"sad is bimodal: {len(low)} low trials (median {np.median(low):.2f}) vs "
            f"{len(high)} high (median {np.median(high):.2f}), gap {gaps[gi]:.2f}. "
            f"Low cluster loses mostly to {rivals.index[0]} ({int(rivals.iloc[0])}/{len(low)}); "
            f"its internal coherence (mean pairwise cosine) is {coh:.2f}. "
            "High coherence (>~0.6) with a consistent rival means the low cluster is a real, "
            "self-consistent production — a SECOND VARIANT of sad (or systematic mislabels toward "
            "that rival). Low coherence means scattered noise: ordinary MISLABELS/bad clips. "
            "Either way these trials should be excluded from centroids or given their own variant "
            "centroid, not averaged in.")
    else:
        sad_para = (f"sad shows no clean bimodality (n={len(scores)}, largest gap "
                    f"{gaps[gi] if len(gaps) else 0.0:.2f}): low scores spread continuously, more "
                    "consistent with diffuse mislabels/bad clips than a coherent second variant.")
    print("\n" + sad_para)

In [ ]:
# diagnosis2_report.md
report2 = "\n\n".join([
    "# Phase 1 diagnostic — round 2",
    f"Loss={cfg.loss}; global threshold {thr:.3f}; val participants {metrics['val_participants']}.",
    "## Task 1 — orientation test (per val participant, both orientations)",
    "TAR columns use the round-1 global threshold, calibrated on STORED-orientation "
    "impostors — descriptive only; verdicts key on the threshold-free genuine medians. "
    "Recalibrate FAR before shipping orientation-max as a fix.",
    md_table(per_pid),
    "Verdicts:\n" + ("\n".join(f"- {v}" for v in verdicts) if verdicts
                     else "- no participant shows an orientation problem"),
    "## Task 2 — label-noise bound",
    f"Suspect genuine trials (score < 0.45, beaten by a non-confusable): "
    f"{int(g.suspect.sum())}/{len(g)} ({g.suspect.mean():.1%}).",
    f"TAR@FAR5: {(g.score >= thr).mean():.1%} all -> **{clean_tar:.1%} clean-TAR bound**.",
    "Signs over 15% suspect rate:",
    md_table(flagged) if len(flagged) else "(none)",
    "## Task 3 — sad deep-dive",
    "```\n" + sad_hist_txt + "\n```",
    sad_para,
])
Path("/kaggle/working/diagnosis2_report.md").write_text(report2 + "\n")
print("wrote /kaggle/working/diagnosis2_report.md")